# PySpark text job (notebook version of `src/08_spark_text_job.py`)

Runs in **local mode** over trailer comments + the IMDb 50K corpus (tagged by `source`). Each stage prints its row count. The functions are imported from the script so the notebook and the script can never disagree.

In [1]:
import sys, importlib
sys.path.insert(0, '../src')
job = importlib.import_module('08_spark_text_job')
spark = job.start_spark()
spark.sparkContext.setLogLevel('ERROR')
spark.version

26/11/26 18:22:43 WARN Utils: Your hostname, Shubhadityas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.6 instead (on interface en0)
26/11/26 18:22:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 18:22:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/26 18:22:43 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


'3.5.3'

## Stage 1 — load and union (must be ≥ 50,000 rows)

In [2]:
df = job.load(spark)
df.groupBy('source').count().show()

[1a load comments            ]      350,426 rows  


[1b load imdb (deduplicated) ]       49,582 rows  


[1c union                    ]      400,008 rows  must be ≥ 50,000


+----------------+------+
|          source| count|
+----------------+------+
|youtube_comments|350426|
|        imdb_50k| 49582|
+----------------+------+



## Stage 2 — clean text (lowercase, keep letters + combining marks of any script, drop empty)

In [3]:
df = job.clean(df)
df.select('source', 'film_id', 'text').show(5, truncate=80)

[2 clean (non-empty)         ]      371,616 rows  emoji-only / number-only texts drop out here


+----------------+-------+--------------------------------------------------------------------------------+
|          source|film_id|                                                                            text|
+----------------+-------+--------------------------------------------------------------------------------+
|youtube_comments|chhaava|                    we need more and more and more films like this please please|
|youtube_comments|chhaava|                                                        akshay shadowed everyone|
|youtube_comments|chhaava|रेहमानला देऊन मोठी चुक केली उत्तेकरांनी अक्कल घाण ठेवली होती का अजय अतुल यांन...|
|youtube_comments|chhaava|                                                     anybody after eetha trailer|
|youtube_comments|chhaava|    almost years since the release but still got goosebumps watching the trailer|
+----------------+-------+--------------------------------------------------------------------------------+
only showing top 5 rows



## Stage 3 — measure length

In [4]:
measured = job.measure(df).cache()
measured.select('source', 'char_length', 'word_count').summary('mean', 'min', 'max').show()

[3 length measured           ]      371,616 rows  


+-------+----------------+------------------+-----------------+
|summary|          source|       char_length|       word_count|
+-------+----------------+------------------+-----------------+
|   mean|            NULL|206.37872696546972|38.54714544045466|
|    min|        imdb_50k|                 1|                1|
|    max|youtube_comments|             13315|             2494|
+-------+----------------+------------------+-----------------+



## Stage 4 — split into terms, remove English + Hinglish stopwords

In [5]:
kept = job.terms(measured)
kept.show(5)

[4a words (exploded)         ]   14,324,736 rows  one row per word, before stopwords


[4b terms (no stopwords)     ]    7,752,016 rows  
+-------+----------------+------+
|film_id|          source|  term|
+-------+----------------+------+
|chhaava|youtube_comments|  need|
|chhaava|youtube_comments| films|
|chhaava|youtube_comments|  like|
|chhaava|youtube_comments|please|
|chhaava|youtube_comments|please|
+-------+----------------+------+
only showing top 5 rows



## Stage 5 — aggregate by film and source; top 10 terms per film

In [6]:
summary, top = job.aggregate(measured, kept)
summary.show(truncate=False)

[5a summary by film+source   ]           11 rows  


[5b term counts              ]      304,749 rows  distinct (film, term) pairs


[5c top terms per film       ]          110 rows  top 10 × films
+----------------+----------------------+-----+----------+----------+
|source          |film_id               |texts|mean_chars|mean_words|
+----------------+----------------------+-----+----------+----------+
|imdb_50k        |imdb_50k              |49582|1249.5    |234.3     |
|youtube_comments|chhaava               |82173|51.5      |9.3       |
|youtube_comments|dhurandhar_the_revenge|50154|48.5      |9.1       |
|youtube_comments|housefull_5           |29898|39.4      |7.5       |
|youtube_comments|kantara_chapter_1     |17221|33.6      |5.7       |
|youtube_comments|l2_empuraan           |27334|37.7      |6.1       |
|youtube_comments|lokah_chapter_1       |4377 |34.8      |5.8       |
|youtube_comments|peddi                 |15749|38.9      |6.6       |
|youtube_comments|raid_2                |26854|31.7      |6.1       |
|youtube_comments|saiyaara              |24658|46.3      |9.0       |
|youtube_comments|sitaare

In [7]:
top_pd = top.toPandas()
top_pd.groupby('film_id')['term'].apply(', '.join).to_frame('top 10 terms')

,top 10 terms
film_id,
chhaava,"vicky, trailer, har, movie, goosebumps, jai, m..."
dhurandhar_the_revenge,"trailer, movie, dhurandhar, toxic, dhar, adity..."
housefull_5,"movie, bro, trailer, comedy, akshay, flop, hou..."
imdb_50k,"movie, film, one, like, good, time, even, stor..."
kantara_chapter_1,"trailer, movie, kantara, wow, best, great, sup..."
l2_empuraan,"trailer, movie, ഇത്, ഒരു, mohanlal, goosebumps..."
lokah_chapter_1,"movie, hindi, trailer, dq, malayalam, lokah, h..."
peddi,"peddi, charan, trailer, movie, jai, ram, super..."
raid_2,"trailer, ajay, riteish, teaser, patnaik, yo, m..."


## Row counts at every stage

In [8]:
import pandas as pd
pd.DataFrame(job.stage_counts)

,stage,rows,note
0,1a load comments,350426,
1,1b load imdb (deduplicated),49582,
2,1c union,400008,"must be ≥ 50,000"
3,2 clean (non-empty),371616,emoji-only / number-only texts drop out here
4,3 length measured,371616,
5,4a words (exploded),14324736,"one row per word, before stopwords"
6,4b terms (no stopwords),7752016,
7,5a summary by film+source,11,
8,5b term counts,304749,"distinct (film, term) pairs"
9,5c top terms per film,110,top 10 × films


In [9]:
spark.stop()